In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("FoodDeliveryConsumer")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "food-delivery-updates")
    .option("startingOffsets", "latest")
    .load()
)

df = df_raw.selectExpr("CAST(value AS STRING) AS value")

In [3]:
schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("restaurant_id", IntegerType()),
    StructField("delivery_area", StringType()),
    StructField("status", StringType()),
    StructField("estimated_delivery_time", IntegerType()),
    StructField("cuisine", StringType()),
    StructField("event_time", StringType())
])

parsed = (
    df.select(from_json(col("value"), schema).alias("data"))
      .select("data.*")
      .withColumn("event_time", to_timestamp("event_time"))
      .filter(col("event_time").isNotNull())
)

display(parsed)

DataFrame[order_id: int, restaurant_id: int, delivery_area: string, status: string, estimated_delivery_time: int, cuisine: string, event_time: timestamp]

In [4]:
agg = (
    parsed
    .withWatermark("event_time", "2 minutes")
    .groupBy(
        window(col("event_time"), "2 minutes"),
        col("delivery_area")
    )
    .count()
)

query = (
    agg.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", False)
    .trigger(processingTime="10 seconds")
    .start()
)

In [5]:
query.stop()

cuisine_agg = (
    parsed
    .withWatermark("event_time", "2 minutes")
    .groupBy(
        window(col("event_time"), "2 minutes"),
        col("delivery_area"),
        col("cuisine")
    )
    .count()
)

query = (
    cuisine_agg.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", False)
    .trigger(processingTime="10 seconds")
    .start()
)

In [6]:
query.stop()

status_agg = (
    parsed
    .withWatermark("event_time", "2 minutes")
    .groupBy(
        window(col("event_time"), "2 minutes"),
        col("delivery_area"),
        col("status")
    )
    .count()
)

query = (
    status_agg.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", False)
    .trigger(processingTime="10 seconds")
    .start()
)